# Limpeza e Estruturação dos Dados de Elenco (Rosters)

**Objetivo:** Ler o arquivo `rosters.csv`, que contém informações de elenco com formatação aninhada em strings, e transformá-lo em um DataFrame limpo e estruturado.

**Processo:**
1.  Ler o arquivo `rosters.csv`.
2.  "Parsear" as colunas `player` e `team` para extrair os IDs.
3.  Selecionar e renomear as colunas desejadas: `game_id`, `player_id`, `shirt`, `team_id`.
4.  Garantir que os tipos de dados estejam corretos (numéricos).
5.  Salvar o resultado como `rosters.parquet`.

In [2]:
import pandas as pd
import os
import ast # Usado para converter strings em dicionários de forma segura

In [3]:
def parse_dict_string(dict_string: str):
    """
    Converte uma string que se parece com um dicionário em um dicionário Python real.
    Usa ast.literal_eval por ser mais seguro que o eval().
    """
    try:
        # Verifica se o valor é uma string antes de tentar o parse
        if isinstance(dict_string, str):
            return ast.literal_eval(dict_string)
        return None # Retorna None se não for uma string (ex: NaN)
    except (ValueError, SyntaxError):
        # Retorna None se a string não for um dicionário válido
        return None

In [10]:
# --- Definir Caminhos ---
rosters_csv_path = 'rosters.csv'
rosters_parquet_path = 'rosters.parquet' # O nome do arquivo de saída

# --- Verificar se o arquivo de entrada existe ---
if not os.path.exists(rosters_csv_path):
    raise FileNotFoundError(f"Arquivo de entrada não encontrado: '{rosters_csv_path}'")

# --- Carregar os Dados Brutos ---
print(f"Carregando dados brutos de '{rosters_csv_path}'...")
df_rosters_raw = pd.read_csv(rosters_csv_path)
print("Dados brutos:")
display(df_rosters_raw.head(3))
print("\nTipos de dados originais:")
df_rosters_raw.info()

# --- Limpeza e Extração ---
print("\nIniciando o processo de limpeza e extração...")

# 1. Aplica a função para converter as strings 'player' e 'team' em dicionários
df_rosters_raw['player_dict'] = df_rosters_raw['player'].apply(parse_dict_string)
df_rosters_raw['team_dict'] = df_rosters_raw['team'].apply(parse_dict_string)

# 2. Extrai o 'id' de cada dicionário. O .str.get() lida com segurança com valores nulos.
df_rosters_raw['player_id'] = df_rosters_raw['player_dict'].str.get('id')
df_rosters_raw['team_id'] = df_rosters_raw['team_dict'].str.get('id')

# 3. Seleciona e renomeia as colunas para criar o DataFrame final
df_rosters_clean = df_rosters_raw[['game_id', 'player_id', 'shirtNumber', 'positionGroupType', 'team_id']].copy()
df_rosters_clean.rename(columns={'shirtNumber': 'shirt'}, inplace=True)

# 4. Converte as colunas para tipo numérico e remove linhas com erro
for col in df_rosters_clean.columns:
    if col in ['game_id', 'player_id', 'shirt', 'team_id']:
        df_rosters_clean[col] = pd.to_numeric(df_rosters_clean[col], errors='coerce')
df_rosters_clean.dropna(inplace=True)

# 5. Converte os tipos de dados para inteiro, pois IDs e camisas são números inteiros
for col in ['game_id', 'player_id', 'shirt', 'team_id']:
    df_rosters_clean[col] = df_rosters_clean[col].astype(int)

# --- Visualizar o Resultado Limpo ---
print("\nLimpeza concluída! Amostra do DataFrame final:")
display(df_rosters_clean.head())
print("\nTipos de dados finais:")
df_rosters_clean.info()

# --- Salvar o Arquivo Final ---
df_rosters_clean.to_parquet(rosters_parquet_path, index=False)
print(f"\nArquivo limpo e estruturado salvo com sucesso em: '{rosters_parquet_path}'")

Carregando dados brutos de 'rosters.csv'...
Dados brutos:


,game_id,player,positionGroupType,shirtNumber,team
0,13335,"{'id': '6056', 'nickname': 'James Trafford'}",GK,1,"{'id': '5', 'name': 'Burnley'}"
1,13335,"{'id': '13068', 'nickname': 'Luca Koleosho'}",RW,30,"{'id': '5', 'name': 'Burnley'}"
2,13335,"{'id': '4880', 'nickname': 'Jacob Bruun Larsen'}",LW,34,"{'id': '5', 'name': 'Burnley'}"



Tipos de dados originais:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11263 entries, 0 to 11262
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   game_id            11263 non-null  int64 
 1   player             11263 non-null  object
 2   positionGroupType  11263 non-null  object
 3   shirtNumber        11263 non-null  int64 
 4   team               11263 non-null  object
dtypes: int64(2), object(3)
memory usage: 440.1+ KB

Iniciando o processo de limpeza e extração...

Limpeza concluída! Amostra do DataFrame final:


,game_id,player_id,shirt,positionGroupType,team_id
0,13335,6056,1,GK,5
1,13335,13068,30,RW,5
2,13335,4880,34,LW,5
3,13335,22,17,CM,11
4,13335,13321,21,LB,11



Tipos de dados finais:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11263 entries, 0 to 11262
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   game_id            11263 non-null  int32 
 1   player_id          11263 non-null  int32 
 2   shirt              11263 non-null  int32 
 3   positionGroupType  11263 non-null  object
 4   team_id            11263 non-null  int32 
dtypes: int32(4), object(1)
memory usage: 264.1+ KB

Arquivo limpo e estruturado salvo com sucesso em: 'rosters.parquet'


In [11]:
df = pd.read_parquet(rosters_parquet_path)
df.head()

,game_id,player_id,shirt,positionGroupType,team_id
0,13335,6056,1,GK,5
1,13335,13068,30,RW,5
2,13335,4880,34,LW,5
3,13335,22,17,CM,11
4,13335,13321,21,LB,11
